In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics

In [ ]:
!unzip /content/drive/MyDrive/EdgeCard_System/4-point-card.zip

In [ ]:
from ultralytics import YOLO

def main():
    model = YOLO("yolo26n-pose.pt")

    print("Bắt đầu huấn luyện YOLO26-Pose cho bài toán bẻ phẳng thẻ...")

    # 2. Bắt đầu huấn luyện
    results = model.train(
        data="/content/4-point-card/data.yaml",
        epochs=100,
        imgsz=640,
        batch=32,

        # --- CÁC SIÊU THAM SỐ TỐI ƯU CHO KEYPOINTS ---
        pose=1.0,                   # Trọng số hàm loss của điểm mốc (Tăng lên để AI tập trung bắt góc thẻ chuẩn hơn)
        kobj=1.0,                   # Trọng số tính sự tồn tại của điểm mốc

        # --- AUGMENTATION (Tăng cường dữ liệu thực tế) ---
        degrees=15.0,               # Xoay ảnh ngẫu nhiên +/- 15 độ
        hsv_s=0.5,                  # Thay đổi độ bão hòa màu (giúp AI quen với thẻ bị chói/tối)
        hsv_v=0.4,                  # Thay đổi độ sáng
        scale=0.5,                  # Phóng to/thu nhỏ ảnh
        close_mosaic=10,

        project="/content/drive/MyDrive/EdgeCard_System/stage_1",
        name="Stage1_YOLO26_Pose-v2"
    )

    print("\n[HOÀN TẤT] Trọng số mô hình tốt nhất đã được lưu.")

if __name__ == "__main__":
    main()

In [ ]:
from ultralytics import YOLO

# Nạp mô hình đã train
model = YOLO("/content/drive/MyDrive/EdgeCard_System/stage_1/Stage1_YOLO26_Pose-v2-2/weights/best.pt")

# Tiến hành đánh giá trên tập Test
metrics = model.val(data="/content/4-point-card/data.yaml", split="test")

# In ra độ đo mAP của Keypoints
print(f"mAP50-95 của 4 góc thẻ (Pose): {metrics.pose.map:.4f}")
print(f"mAP50 của 4 góc thẻ (Pose): {metrics.pose.map50:.4f}")